In [1]:
import sys, os

os.environ.pop("PYTHONPATH", None)
os.environ.pop("PYTHONHOME", None)

sys.path = [
    p for p in sys.path
    if "/mnt/modules/easybuild" not in p
]

import typing_extensions
print(typing_extensions.__file__)

sys.path.insert(
    0,
    "/data/groups/vib.ai/stein.aerts/gpartel/software/deepSCENIC-dev/src"
)

/data/groups/vib.ai/stein.aerts/gpartel/software/miniconda3/envs/deepSCENIC/lib/python3.11/site-packages/typing_extensions.py


# Tutorial 2: Training

This tutorial covers training the deepSCENIC model to learn gene regulatory networks from multiome data.

```{note}
This part of the tutorials takes a long time and is best run as a SLURM job from a separate python script.
```

## Setup

First, let's import the required packages and check GPU availability:

In [2]:
import torch

import deepscenic as ds

print(f"deepSCENIC version: {ds.__version__}")

# Check GPU availability
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")
if device == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")

deepSCENIC version: 0.1.0
Using device: cuda
GPU: NVIDIA H100 80GB HBM3


```{note}
Training deepSCENIC requires a GPU with at least 32GB VRAM recommended. Training on CPU is technically possible but extremely slow (hours vs minutes per epoch). See the "Custom Sequence Models" section below if you need a lighter alternative.
```

## Load Preprocessed Data

We'll use the preprocessed MuData from Tutorial 1. The data must contain:
- RNA and ATAC modalities with matching cells
- TF annotations (`is_tf` column in `rna.var`)
- Region-to-gene search space (`r2g` in `uns`)
- Train/test split (`split` column in `obs` and `var`)

In [3]:
# Load preprocessed MuData from previous tutorial
import os

# data_dir = "./data/MM_lines/"
data_dir = "/data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/"
assert os.path.exists(data_dir), "dir not found"

mdata = ds.read(data_dir + "preprocessed_data.h5mu")
print(
    f"Cells: {mdata.n_obs} ({(mdata.obs['split'] == 'train').sum()} train, {(mdata.obs['split'] == 'test').sum()} test)"
)
print(f"Genes: {mdata['rna'].n_vars} ({mdata['rna'].var['is_tf'].sum()} TFs)")
mdata

deepscenic.io - INFO - Loaded /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/preprocessed_data.h5mu: 936 cells, 12109 genes, 288643 regions


Cells: 936 (748 train, 188 test)
Genes: 12109 (1446 TFs)


MuData object with n_obs × n_vars = 936 × 300752
  obs:	'cell_line', 'cell_state', 'split'
  uns:	'r2g'
  2 modalities
    rna:	936 x 12109
      var:	'n_cells', 'is_tf', 'chromosome', 'tss', 'split'
      uns:	'log1p', 'tf_order'
    atac:	936 x 288643
      var:	'chromosome', 'start', 'end', 'split'

## Register Genome

deepSCENIC extracts DNA sequences from genomic regions to learn TF binding patterns. Before training, you must register a genome FASTA file that matches your data's reference assembly.

```{note}
You can also use this `genome` object to fetch fasta sequences. See the API documentation.
```

In [4]:
try:
    genome = ds.get_genome()
    print(f"Genome already registered: {genome.name}")
except RuntimeError:
    # ds.register_genome("./data/hg38.fa")
    ds.register_genome("/data/groups/vib.ai/stein.aerts/gpartel/resources/hg38.fa")

deepscenic.genome - INFO - Registered genome: hg38 (/data/groups/vib.ai/stein.aerts/gpartel/resources/hg38.fa), chromosomes with 'chr' prefix


In [5]:
# Filter ATAC regions to chromosomes present in the genome FASTA.
# This removes regions on unplaced scaffolds or patches not in your FASTA file.
ds.pp.filter_by_genome(mdata)

deepscenic.pp - INFO - Removed 91 ATAC regions on chromosomes not found in genome (11 chromosomes): ['GL000009.2', 'GL000194.1', 'GL000195.1', 'GL000205.2', 'GL000218.1', 'GL000219.1', 'KI270711.1', 'KI270713.1', 'KI270726.1', 'KI270727.1', 'KI270728.1']
deepscenic.pp - INFO - Updated r2g matrix: 288552 regions, 3245122 links remaining


## Understanding Training Configuration

Training is configured through the `ds.tl.train()` function, which accepts both direct parameters and, optionally, a `ModelConfig` object. Understanding the key parameters helps you adapt training to your specific dataset and hardware.

### Two Ways to Configure Training

1. **Direct parameters** (simple): Pass common settings directly to `ds.tl.train()`
2. **ModelConfig** (advanced): Use a config object for full control over all model hyperparameters (e.g. hidden layer size)

### Key Parameters

The most important parameters to adjust:

| Parameter | Default | When to change |
|-----------|---------|----------------|
| `epochs` | 100 | Increase for better convergence |
| `batch_size` | 16 | Reduce if GPU memory errors |
| `seq_batch_size` | 200 | Reduce if GPU memory errors |
| `lr` | 1e-4 | Lower if training unstable, increase if training too slow |

## Train the Model

Now we're ready to train. The `ds.tl.train()` function initializes all model components, computes the initial tf2r matrix from DNA sequences, and runs the training loop.

```{note}
The first training run downloads the pretrained Enformer model (~1GB). Subsequent runs use the cached version. Initial tf2r computation also takes several minutes as it processes all genomic sequences.
```

In [ ]:
# Train the deepSCENIC model
# For demonstration we use 100 epochs; monitor your loss curves and keep training until the model converges
model = ds.tl.train(
    mdata,
    epochs=1000,
    batch_size=16,
    seq_batch_size=200,
    lr=1e-4,
    atac_tau=2,
    device=device,
    checkpoint_dir=data_dir + "checkpoints/first_phase/",  # save best.pt checkpoints during training
    logger="wandb",  # or 'dict', 'tensorboard'. Install what you need.
    resume_from=None,  # set this path to a checkpoint to resume training from an earlier run
)
print("Training complete!")

deepscenic.tl - INFO - Using device: cuda


deepscenic.tl - INFO - Using 11582/12109 genes for reconstruction (train split)
deepscenic.tl - INFO - Using 279944/288552 regions for ATAC reconstruction (train split)
deepscenic.tl - INFO - R2G sparsity on 3081573/3245122 links (train split)
deepscenic.tl - INFO - Test chromosome evaluation: 8608 test regions


Init tf2r (train):   0%|          | 0/1400 [00:00<?, ?it/s]

Init tf2r (test):   0%|          | 0/44 [00:00<?, ?it/s]

deepscenic.tl - INFO - Starting training: 1000 epochs, 936 cells, 12109 genes, 1446 TFs, 288552 regions
deepscenic.tl - INFO - Best checkpoint saving enabled: saving to 'best.pt' on improvement


...

deepscenic.tl - INFO - New best val_chrom/atac_recon (checkpoint): 0.805233 - saving to /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/checkpoints/first_phase/best.pt


Epoch 71/1000:   0%|          | 0/47 [00:00<?, ?it/s]

deepscenic.tl - INFO - New best val_chrom/atac_recon (checkpoint): 0.804802 - saving to /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/checkpoints/first_phase/best.pt


Epoch 72/1000:   0%|          | 0/47 [00:00<?, ?it/s]

Epoch 73/1000:   0%|          | 0/47 [00:00<?, ?it/s]

In [9]:
# model = ds.tl.load_model(data_dir + "checkpoints/first_phase/best.pt", device=device)

deepscenic.tl - INFO - Loaded model from /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/checkpoints/first_phase/best.pt: 12109 genes, 1446 TFs, 288552 regions
deepscenic.tl - INFO -   Training state: epoch=320, best_loss=0.728217


## Phase 2: Fine-tune the r2g Matrix

Phase 2 fine-tunes the r2g (region→gene) matrix in a single process. It first
recomputes tf2r for all regions because Phase 1 updates its cache incrementally.
It then freezes the rest of the model and jointly refits all r2g links by
reconstructing all genes.

Finetuning preserves the cell split: training cells are used for optimization
and test cells are used for validation. Set `balance_class=True` together with
`class_key` to sample training cells with class-balanced weights.

Since we're only fine-tuning the r2g matrix, the training loop itself is fast.
The tf2r recomputation at the start takes a few minutes as it runs
all regions through the sequence model. Best and periodic checkpoints are complete
model files that can be loaded directly with {func}`~deepscenic.tl.load_model`.

See {func}`~deepscenic.tl.finetune_r2g` for full parameter reference.

In [10]:
# Phase 2: Refit all r2g links with a freshly recomputed tf2r matrix
model = ds.tl.finetune_r2g(
    model,
    mdata,
    epochs=1000,
    lr=1e-4,
    batch_size=16,
    use_scheduler=False,  # enable to reduce LR when validation loss plateaus
    reinit_r2g=True,  # reinitialize all r2g links before finetuning
    checkpoint_dir=data_dir + "checkpoints/second_phase/",
    logger="wandb",
    logger_kwargs={
        "project": "deepscenic",
        "name": "r2g_finetuning",
    },
)

deepscenic.tl - INFO - Using device: cuda


deepscenic.tl - INFO - Recomputing tf2r matrix from sequence model...


Recomputing tf2r:   0%|          | 0/1443 [00:00<?, ?it/s]

deepscenic.tl - INFO - Using all 12109 genes for reconstruction
deepscenic.tl - INFO - Using all 3245122 r2g links for finetuning
deepscenic.tl - INFO - R2G finetuning: all genes, regions, and links
deepscenic.tl - INFO - Best checkpoint saving enabled: saving to 'finetune_best.pt' on improvement
deepscenic.tl - INFO - Starting R2G finetuning: 1000 epochs


Finetune R2G 1/1000:   0%|          | 0/47 [00:00<?, ?it/s]

...

Finetune R2G 998/1000:   0%|          | 0/47 [00:00<?, ?it/s]

Finetune R2G 999/1000:   0%|          | 0/47 [00:00<?, ?it/s]

Finetune R2G 1000/1000:   0%|          | 0/47 [00:00<?, ?it/s]

train/r2g_l1,▁▂▃▃▃▄▄▄▄▅▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇▇▇▇██████████
train/rna_recon,█▃▃▃▂▂▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁▁
train/total,█▅▅▅▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▁▂▁▁▁▁▁▁▁▁▁▁▁
val_cells/total,█▇▇▆▅▄▃▃▃▃▃▃▂▂▂▂▂▂▂▂▂▂▂▂▂▂▁▂▁▂▁▁▁▁▁▁▁▁▁▁
train/r2g_l1,0.00086
train/rna_recon,0.10375
train/total,0.10461
val_cells/total,0.10507


deepscenic.tl - INFO - R2G finetuning completed


## Monitor Training Progress

deepSCENIC supports logging backends for tracking training metrics:

```python
# TensorBoard logging
model = ds.tl.train(
    mdata,
    epochs=100,
    logger="tensorboard",
    log_dir="./runs/deepscenic_experiment",
    device=device,
)
```

Then launch TensorBoard in a separate terminal:
```bash
tensorboard --logdir ./runs
```

Alternatively, use Weights & Biases with `logger="wandb"`. You can set optional logger kwargs with the `logger_kwargs` argument (e.g. `logger_kwargs={project="deepscenic"}`) which will be passed to the `wandb.init()`.

## Save and Load Models

Trained models can be saved for later use. The saved file includes:
- All model weights (VAE, TF2rNet, sequence model)
- The tf2r matrix cache
- Training configuration
- Feature names (TFs, genes, regions)

```{note}
Model files are large (~1GB+) when using Enformer. Models trained with custom sequence models will be smaller.
Also, checkpoints follow the same format as models saved explicitly with `model.save`, so you can load these in the exact same way.
```

In [11]:
# Save the finetuned model
model.save(data_dir + "checkpoints/deepscenic_model.pt")

deepscenic.tl - INFO - Saved model to /data/groups/vib.ai/stein.aerts/gpartel/data/MM_lines_demo/checkpoints/deepscenic_model.pt


In [8]:
# Load a saved model in future sessions
model = ds.tl.load_model(data_dir + "checkpoints/deepscenic_model.pt", device=device)

/data/groups/vib.ai/stein.aerts/lmahieu/software/miniconda3/envs/deepscenic-dev/lib/python3.13/site-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
deepscenic.tl - INFO - Loaded model from ./data/MM_lines/checkpoints/deepscenic_model.pt: 12158 genes, 1463 TFs, 288552 regions


## Custom Sequence Models

By default, deepSCENIC uses Enformer as its sequence encoder. However, you can provide your own PyTorch module instead. This is useful when:

- You have a custom architecture tailored to your genomic context
- You want to experiment with different sequence representations

### Model Requirements

Your custom sequence model must satisfy these requirements:

| Requirement | Details |
|-------------|---------|
| **Input shape** | `(batch, seq_len, 4)` - one-hot encoded DNA sequences |
| **Output shape** | `(batch, bottleneck_size * emb_len)` - flattened embeddings |
| **Config sync** | Set matching `seq_len`, `bottleneck_size`, `emb_len` in config |

The three key parameters control the model's interface:

- **`seq_len`**: Input DNA sequence length in base pairs. Default is 640 for Enformer compatibility. Set this to match your model's expected input length.
- **`bottleneck_size`**: Embedding dimension per position. Default is 3072 (Enformer's embedding size). This is the number of features your model outputs per position.
- **`emb_len`**: Number of output positions. Default is 5 (Enformer's target_length). Combined with bottleneck_size, this determines your model's total output: `bottleneck_size * emb_len`.

In [10]:
import torch.nn as nn

from deepscenic.tl import ModelConfig


# Example: A lightweight convolutional sequence model
class LightweightSequenceModel(nn.Module):
    """Lightweight custom sequence encoder."""

    def __init__(self, seq_len=640, bottleneck_size=256, emb_len=60):
        super().__init__()
        # Store dimensions for verification
        self.seq_len = seq_len
        self.bottleneck_size = bottleneck_size
        self.emb_len = emb_len
        self.output_dim = bottleneck_size * emb_len

        # Simple conv architecture
        self.conv = nn.Sequential(
            nn.Conv1d(4, 64, kernel_size=15, padding=7),
            nn.ReLU(),
            nn.Conv1d(64, bottleneck_size, kernel_size=15, padding=7),
            nn.ReLU(),
        )
        self.pool = nn.AdaptiveAvgPool1d(emb_len)

    def forward(self, x):
        """Transform one-hot DNA to flattened embeddings."""
        # x: (batch, seq_len, 4)
        x = x.transpose(1, 2)  # (batch, 4, seq_len)
        x = self.conv(x)  # (batch, bottleneck_size, seq_len)
        x = self.pool(x)  # (batch, bottleneck_size, emb_len)
        return x.flatten(1)  # (batch, bottleneck_size * emb_len)


# Create model instance
custom_model = LightweightSequenceModel(
    seq_len=640,
    bottleneck_size=256,
    emb_len=60,
)

# Configure the model architecture with matching dimensions
config = ModelConfig(
    sequence_model=custom_model,
    seq_len=640,  # Must match model's expected input
    bottleneck_size=256,  # Must match model's output channels
    emb_len=60,  # Must match model's output length. Usually this is =1. Enformer uses 5 (5 bins of 128bp).
)

print(f"Custom model output: {custom_model.output_dim} = {config.bottleneck_size} x {config.emb_len}")

Custom model output: 15360 = 256 x 60


### Training with Custom Models

```python
# Train with custom sequence model
model = ds.tl.train(mdata, config=config, device=device)

# Save the model
model.save("custom_model.pt")
```

### Loading Custom Models

When loading a model trained with a custom sequence model, you must provide an instance of the same architecture:

```python
# Recreate the model architecture (same as during training)
custom_model = LightweightSequenceModel(seq_len=640, bottleneck_size=256, emb_len=60)

# Load with the custom model
loaded_model = ds.tl.load_model(
    "custom_model.pt",
    device=device,
    sequence_model=custom_model,  # Required for custom models
)
```

```{warning}
The model architecture must exactly match what was used during training. Only the weights are saved in the checkpoint file, not the model class definition.
```

## Next Steps

Now that you have a trained model, you can:

- Diagnose model training and inspect results: {doc}`03_model_diagnosis`
- Extract and analyze the learned GRN: {doc}`04_grn_analysis`
- Simulate perturbation experiments: {doc}`05_perturbation_analysis`